In [1]:
from ppopt.mplp_program import MPLP_Program
from ppopt.mpmodel import MPModeler
from ppopt.mp_solvers.solve_mpqp import solve_mpqp, mpqp_algorithm
from typing import List, Tuple, Callable, Union, Dict, Any, Optional
from collections import defaultdict
from numpy.polynomial.legendre import leggauss
from scipy.optimize import linprog
from sympy.core.relational import Relational as SympyRelational
import itertools as itools
from sympy.logic.boolalg import BooleanTrue, BooleanFalse
import numpy as np
import chaospy as cp
import math
import pickle
from pathlib import Path
import sympy as sp
import time
import pandas as pd
import warnings
import gurobipy as gp
from gurobipy import GRB
from gurobipy import nlfunc
import matplotlib.pyplot as plt

## Smolyak quadrature

In [2]:
def theta_interval_at_point(solution, theta_vector: np.ndarray, max_idx: int = 0, min_idx: int = 1) -> tuple:
    """Given the parametric solution for theta_k and the current 'state' vector (theta_prev + d),
        return the scalar lower and upper bound [t_min, t_max] for this theta_k.

    Args:
        solution (_type_): _description_
        t_vector (np.ndarray): _description_
        max_idx (int, optional): _description_. Defaults to 0.
        min_idx (int, optional): _description_. Defaults to 1.

    Returns:
        tuple: _description_
    """

    theta_vector_aug = np.append(theta_vector, 1).reshape(-1, 1)

    if isinstance(solution, list):
        theta_min = solution[0]
        theta_max = solution[1]
        return float(theta_min), float(theta_max)
    
    try:
        region = solution.get_region(theta_vector.reshape(-1, 1))
        coefficients = np.concatenate([region.A, region.b], axis=1)[:2, :]
        max_coefficients = coefficients[max_idx]
        min_coefficients = coefficients[min_idx]
        # theta_max = float(max_coefficients @ theta_vector_aug)
        # theta_min = float(min_coefficients @ theta_vector_aug)
        theta_max = (max_coefficients @ theta_vector_aug).item()
        theta_min = (min_coefficients @ theta_vector_aug).item()
        return theta_min, theta_max
    except:
        raise ValueError(
        "The provided theta_vector is not inside any critical region of the solution.")

def map_u_to_theta_and_jacobian(solutions, u, dvector):
    """
    Given stage-aligned parametric solutions for theta_k and the current
    disturbance vector d, return the theta vector and the Jacobian.

    Parameters
    ----------
    solutions : list
        Stage-aligned list. Each entry is either:
        - a valid parametric solution object for stage k, or
        - None for an empty / unusable stage.
    u : np.ndarray
        1D array of canonical coordinates.
    dvector : np.ndarray
        Current disturbance/design vector.

    Returns
    -------
    theta_values : np.ndarray
        Stage-aligned theta vector, same length as solutions.
    jacobian : float
    """
    theta_values = []
    jacobian = 1.0

    for k, sol in enumerate(solutions):
        if sol is None:
            theta_k = 0.0
            theta_values.append(theta_k)
            continue

        if isinstance(dvector, np.ndarray):
            thetavector = np.block([np.array(theta_values, dtype=float), dvector])
        else:
            thetavector = np.array(theta_values, dtype=float)

        thetamin, thetamax = theta_interval_at_point(sol, thetavector)
        length = thetamax - thetamin

        theta_k = 0.5 * length * u[k] + 0.5 * (thetamax + thetamin)
        theta_values.append(theta_k)

        jacobian *= 0.5 * length

    return np.array(theta_values, dtype=float), jacobian


def calculate_stocflexibility_smolyak(solutions: List, level: int, joint_func: Callable[[List[float]], float], d_vector: np.ndarray = None, rule: str = "gaussian", ns_u = None, ws_u = None) -> float:
    """Compute stochastic flexibility using a Smolyak sparse grid in canonical  u-space

    Args:
        solutions (List): list of solutions for each theta dimension (same structure as in calculate_stocflexibility)
        level (int): Smolyak level (1,2,3,...) controls accuracy & number of points
        joint_func (Callable[[List[float]], float]): callable f(theta_list) -> scalar
        d_vector (np.ndarray, optional):design vector (np.ndarray). Defaults to None.
        rule (str, optional): 1D quadrature rule passed to chaospy (e.g. "gaussian"). Defaults to "gaussian".

    Returns:
        float: _description_
    """

    n_theta = len(solutions)
    
    if ns_u is None or ws_u is None:
        ns_u, ws_u = smolyak_nodes_weights(
            n_theta=n_theta,
            level=level,
            rule=rule,
            growth=True,
        )

    start = time.time()
    stochastic_flexibility = 0.0

    for i in range(ns_u.shape[0]):
        u_vector = ns_u[i, :]
        theta_vector, jacobian = map_u_to_theta_and_jacobian(
            solutions, u_vector, d_vector)
        func_value = joint_func(theta_vector)
        stochastic_flexibility += func_value * jacobian * ws_u[i]

    end = time.time()
    print(
        f"Smolyak stochastic flexibility computed in {end - start:.4f} seconds.")
    return stochastic_flexibility

## Gaussian Legendre quadrature

In [3]:
def gauss_legendre_between_bounds(expr_coeffs: np.ndarray, n_gl: int, max_idx: int = 0, min_idx: int = 1):
    """
    Generate n Gauss–Legendre quadrature points and weights between min and max bounds
    defined by two linear expressions.

    Parameters:
        expr_coeffs (np.ndarray): 2xD array. Row 0 = max point co`efficients, Row 1 = min.
        n (int): Number of quadrature points.

    Returns:
        points (np.ndarray): (n, D) array of quadrature points.
        weights (np.ndarray): (n,) array of weights.
    """
    if expr_coeffs.shape[0] != 2:
        raise ValueError("expr_coeffs must have two rows")

    max_coeffs = expr_coeffs[max_idx]
    min_coeffs = expr_coeffs[min_idx]

    # Get Gauss–Legendre points and weights on [-1, 1]
    nodes, weights = leggauss(n_gl)
    weights = weights.reshape(-1, 1)

    # Affine transformation to domain [min_coeffs, max_coeffs]
    points = 0.5 * (np.outer((nodes + 1), max_coeffs) +
                    np.outer((1 - nodes), min_coeffs))

    # Adjust weights to match new domain
    weights = 0.5 * weights@(max_coeffs - min_coeffs).reshape(1, -1)

    return points, weights


def get_quadrature_points(solution, nq: int, t_vector: np.ndarray):
    # Augment t_vector once
    t_vector_aug = np.append(t_vector, 1).reshape(-1, 1)

    if isinstance(solution, list):
        qpoints, qweights = np.polynomial.legendre.leggauss(nq)
        min, max = solution[0], solution[1]
        qps_mapped = 0.5*(max*(1+qpoints) + min*(1-qpoints))
        qws_mapped = 0.5*(max-min)*qweights
        # print(max, min, qps_mapped, qws_mapped)
        return max, min, qps_mapped, qws_mapped
    
    try:
        region = solution.get_region(t_vector.reshape(-1, 1))
        coeffs = np.concatenate([region.A, region.b], axis=1)[:2, :]
        qpoints, qweights = gauss_legendre_between_bounds(
            expr_coeffs=coeffs, n_gl=nq)
        return coeffs[0] @ t_vector_aug, coeffs[1] @ t_vector_aug, qpoints @ t_vector_aug, qweights @ t_vector_aug
    except:
        raise ValueError("No region found that contains the given t_vector.")


def calculate_stocflexibility(
    sols,
    nq: Union[int, list],
    joint_func,
    d_vector: np.ndarray = None,
    verbose: bool = True,
):
    # Validate nq if it's a list
    if isinstance(nq, list):
        if len(nq) != len(sols):
            raise ValueError(
                "If nq is a list, it must have the same length as sols")

    start_time = time.perf_counter()

    def recurse(level: int, theta_prev: list, weight_prev: float) -> float:
        if level == len(sols):
            return weight_prev * joint_func(theta_prev)

        nql = nq[level] if isinstance(nq, list) else nq

        t_vector = (
            np.block([np.array(theta_prev), d_vector])
            if isinstance(d_vector, np.ndarray)
            else np.array(theta_prev)
        )

        _, _, t_points, t_weights = get_quadrature_points(
            solution=sols[level],
            nq=nql,
            t_vector=t_vector,
        )

        t_points = t_points.flatten()
        t_weights = t_weights.flatten()

        return sum(
            recurse(level + 1, theta_prev + [v], weight_prev * w)
            for v, w in zip(t_points, t_weights)
        )

    stflex = recurse(level=0, theta_prev=[], weight_prev=1.0)

    end_time = time.perf_counter()

    if verbose:
        print(f"Gaussian Legendre Stochastic Flexibility Elapsed time: {end_time - start_time:.4f} s"
              )

    return stflex

In [4]:
def mpformulate_theta_bounds(flex_sol, num_theta: int, theta_bounds: list, num_design: int = 0, design_bounds: list = None, psi_idx: int = 0, theta_m: int = 0):
    A0, b0, F0 = np.empty((len(flex_sol), num_theta)), np.empty(
        (len(flex_sol), 1)), np.empty((len(flex_sol), num_design))
    num_cr = len(flex_sol.critical_regions)
    for i, region in enumerate(flex_sol.critical_regions):
        A0[i] = region.A[psi_idx, :num_theta]
        b0[i] = -region.b[psi_idx]
        F0[i] = -region.A[psi_idx, num_theta:num_theta+num_design]
    # print(f'num_cr:{num_cr}')
    # print(f'num_theta:{num_theta}')
    # print(f'num_design:{num_design}')
    # print(f"A0: {A0}")
    # print(f"b0: {b0}")
    # print(f"F0: {F0}")

    c = np.hstack([np.array([-1, 1]).reshape(1, -1),
                  np.zeros((1, 2 * (num_theta - 1 - theta_m)))]).reshape(-1, 1)
    # print(f'c:{c}')
    # print(f'c.shape: {c.shape}')

    row1_block = np.hstack([block for i in range(theta_m, num_theta)
                           for block in (A0[:, [i]], np.zeros((num_cr, 1)))])
    row2_block = np.hstack([block for i in range(theta_m, num_theta)
                           for block in (np.zeros((num_cr, 1)), A0[:, [i]])])
    bound_row = np.hstack([np.array([-1, 1]).reshape(1, -1),
                          np.zeros((1, 2 * (num_theta - 1 - theta_m)))])
    A = np.vstack([row1_block, row2_block, bound_row, -
                  np.eye(2*(num_theta-theta_m)), np.eye(2*(num_theta-theta_m))])
    # print(f'A: {A}')
    # print(f'A.shape: {A.shape}')

    x_lb = np.array([val for i in range(theta_m, len(theta_bounds))
                    for val in [theta_bounds[i][0]] * 2])
    x_ub = np.array([val for i in range(theta_m, len(theta_bounds))
                    for val in [theta_bounds[i][1]] * 2])
    b = np.vstack([b0, b0, np.zeros((1, 1)), -
                  x_lb.reshape(-1, 1), x_ub.reshape(-1, 1)])
    # print(f'b: {b}')
    # print(f'b.shape: {b.shape}')

    if F0.size == 0 and theta_m == 0:
        # print('here')
        return A, b, c, np.array([]), np.array([]), np.array([]), np.array([])

    F = np.vstack([F0, F0, np.zeros((1, num_design)), np.zeros(
        (4*(num_theta-theta_m), num_design))]) if num_design > 0 else np.vstack([F0, F0])
    # print(f'F:{F}')
    # print(f'F.shape: {F.shape}')
    if theta_m > 0:
        F_lltheta = np.hstack([A0[:, [i]] for i in range(theta_m)])
        # print(f'F_lltheta: {F_lltheta}')
        # print(f'F_lltheta.shape: {F_lltheta.shape}')
        F = np.hstack([np.vstack([-F_lltheta, -F_lltheta, np.zeros((1, len(range(theta_m)))), np.zeros((4*(num_theta-theta_m), theta_m))]), F]
                      ) if F.size > 0 else np.vstack([-F_lltheta, -F_lltheta, np.zeros((1, len(range(theta_m)))), np.zeros((4*(num_theta-theta_m), theta_m))])
    # print(f'F:{F}')
    # print(f'F.shape: {F.shape}')

    H = np.zeros((2*(num_theta-theta_m), theta_m+num_design))
    # print(f'H:{H}')
    # print(f'H.shape: {H.shape}')

    A_t = np.vstack([-np.eye(theta_m+num_design), np.eye(theta_m+num_design)])
    # print(f'A_t:{A_t}')
    # print(f'A_t.shape: {A_t.shape}')

    theta_lb = np.array([-theta_bounds[i][0] for i in range(theta_m)] + ([-j[0] for j in design_bounds] if isinstance(design_bounds, list)
                                                                         else [])).reshape(-1, 1)
    theta_ub = np.array([theta_bounds[i][1] for i in range(theta_m)] + ([j[1] for j in design_bounds] if isinstance(design_bounds, list)
                                                                        else [])).reshape(-1, 1)

    b_t = np.vstack([theta_lb, theta_ub])
    # print(f'b_t:{b_t}')
    # print(f'b_t.shape: {b_t.shape}')

    return A, b, c, H, A_t, b_t, F

In [5]:
def get_theta_bounds(flex_sol, numt, tbounds, numd: int = 0, dbounds: list = None, mp_algo: mpqp_algorithm = mpqp_algorithm.combinatorial):
    theta_bound_dict = defaultdict(dict)
    prob_dict = defaultdict(dict)

    for i in range(numt):
        A, b, c, H, A_t, b_t, F = mpformulate_theta_bounds(
            flex_sol=flex_sol, num_theta=numt, num_design=numd, theta_bounds=tbounds, design_bounds=dbounds, theta_m=i)
        # print(f'A.shape:{A.shape}')
        # print(f'b.shape: {b.shape}')
        # print(f'F.shape: {F.shape}')
        if F.size != 0:
            try:
                with warnings.catch_warnings():
                    warnings.filterwarnings("error", category=UserWarning, message="The chebychev ball has either a radius of zero, or the problem is not feasible!")
                    prob = MPLP_Program(A=A, b=b, c=c, H=H, A_t=A_t, b_t=b_t, F=F)

                prob.process_constraints()
                solution = solve_mpqp(problem=prob, algorithm=mp_algo)

                prob_dict[f"t{i}"] = prob
                theta_bound_dict[f"t{i}"] = solution

            except UserWarning as w:
                # MPLP infeasible / degenerate
                print(f"[theta {i}] MPLP infeasible / zero Chebyshev ball: {w}")
                prob_dict[f"t{i}"] = None
                theta_bound_dict[f"t{i}"] = None

        else:
            # LP fallback branch via scipy.optimize.linprog
            linres = linprog(c=c, A_ub=A, b_ub=b)

            if not linres.success:
                print(f"[theta {i}] linprog failed: status={linres.status}, "f"message={linres.message}")
                prob_dict[f"t{i}"] = linres
                theta_bound_dict[f"t{i}"] = None
            else:
                prob_dict[f"t{i}"] = linres
                theta_bound_dict[f"t{i}"] = [linres.x[1], linres.x[0]]

        print(f"Finished solving for theta{i+1}")

    probs = [p for _, p in prob_dict.items()]
    sols = [sol for _, sol in theta_bound_dict.items()]
    return probs, sols

In [6]:
def get_bounds_regions(sols: List, min_idx: int = 1, max_idx: int = 0):
    t_bounds_list = []
    t_regions_list = []

    for theta_sol in sols:    
        # if theta_sol is None:
        #     t_bounds_list.append(np.array([]))  # placeholder, will be ignored
        #     t_regions_list.append(np.array([np.zeros((1, 3))], dtype=float))
        #     continue
        # 
        # if not getattr(theta_sol, "critical_regions", None):
        #     tmin, tmax = float(theta_sol[0]), float(theta_sol[1])
        #     
        #     min_row = [0.0, 0.0, tmin]   # placeholder structure
        #     max_row = [0.0, 0.0, tmax]
        #     
        #     t_bounds_list.append(np.array([[min_row, max_row]], dtype=float))
        #     t_regions_list.append(np.array([np.zeros((1, 3))], dtype=float))
        #     
        #     continue
        
        if theta_sol is None:
            t_bounds_list.append(None)
            t_regions_list.append(None)
            continue
        
        if not getattr(theta_sol, "critical_regions", None):
            t_bounds_list.append(None)
            t_regions_list.append(None)
            continue
        
        min_max_list = []
        region_list = []

        for cr in theta_sol.critical_regions:
            # Store bounds
            Ab = np.concatenate([cr.A, cr.b], axis=1)[:2]
            min_max_list.append([Ab[min_idx].tolist(), Ab[max_idx].tolist()])

            # Store region constraints
            Ef = np.concatenate([cr.E, -cr.f], axis=1)
            region_array = np.array([row.tolist() for row in Ef], dtype=float)
            region_list.append(region_array)

        # Append per-theta data
        t_bounds_list.append(np.array(min_max_list))
        # <-- each region is a 2D array
        t_regions_list.append(np.array(region_list, dtype=object))

    return t_bounds_list, t_regions_list


def generate_region_combos(region_sizes, n_gl):
    """Generate region index combinations based on critical region structure."""
    n_theta = len(region_sizes)
    region_combo_shape = []
    for k in range(n_theta):
        n_paths = int(np.prod(n_gl[:k])) if k > 0 else 1
        region_combo_shape.extend([range(region_sizes[k])] * n_paths)
    return list(itools.product(*region_combo_shape))


def affine_expr(coeffs, symbols):
    return sum(c * s for c, s in zip(coeffs[:-1], symbols)) + coeffs[-1]


def normalized_lhs(ineq):
    return ineq.lhs.expand() if hasattr(ineq, 'lhs') else None

In [7]:
def _prepare_state_data(state, tbounds, dbounds, *, solve_algo, theta_algo, log=False):
    """
    Build and solve the flexibility problem for one state, then return only valid theta-related data.

    Returns
    -------
    dict with keys:
        state
        flex_sol
        sol_list
        t_bounds_list
        t_regions_list
        filtered_solutions
        filtered_theta_bounds
        filtered_theta_regions
    or None if no valid theta regions exist.
    """
    state_model = create_flexibility_model(y_list=state, tbounds=tbounds, dbounds=dbounds)
    state_prob = state_model.formulate_problem()
    state_prob.process_constraints()

    flex_sol = solve_mpqp(problem=state_prob, algorithm=solve_algo)

    if log:
        print(f"Number of critical regions in for flexibility function for state {state}: {len(flex_sol.critical_regions)}")

    _, sol_list = get_theta_bounds(flex_sol=flex_sol, numt=nt, numd=nd, tbounds=t_bounds, dbounds=dbounds, mp_algo=theta_algo)

    t_bounds_list, t_regions_list = get_bounds_regions(sols=sol_list)
    
    filtered_solutions      = []
    filtered_theta_bounds   = []
    filtered_theta_regions  = []
    
    for k, (sol, tb, tr) in enumerate(zip(sol_list, t_bounds_list, t_regions_list)):
        if sol is None or tb is None or tr is None:
            # mark as empty stage but preserve position
            filtered_solutions.append(None)
            filtered_theta_bounds.append(None)
            filtered_theta_regions.append(None)
        else:
            filtered_solutions.append(sol)
            filtered_theta_bounds.append(tb)
            filtered_theta_regions.append(tr)

    return {
        "state": state,
        "flex_sol": flex_sol,
        "sol_list": sol_list,
        "theta_bounds_list": t_bounds_list,
        "theta_regions_list": t_regions_list,
        "filtered_solutions": filtered_solutions,
        "filtered_theta_bounds": filtered_theta_bounds,
        "filtered_theta_regions": filtered_theta_regions,
    }

In [8]:
def _safe_sf_call(func, state, label, **kwargs):
    """
    Safely evaluate a stochastic flexibility routine for one state.

    Returns
    -------
    float or None
        None means: skip this state's contribution.
    """
    try:
        return func(**kwargs)

    except ValueError as e:
        msg = str(e)

        known_skip_markers = (
            "not inside any critical region",
            "No region found that contains the given t_vector",
            "No solution available for this stage",
            "no valid theta regions",
            "no valid stages",
        )

        if any(marker in msg for marker in known_skip_markers):
            print(f"Skipping {label} SF for state {state}: {e}")
            return None

        raise

In [9]:
def calculate_gl_esf(y_d: dict, prepared_data_by_state: dict, d_v, n_q: int):
    """
    Gaussian-Legendre ESF using stage-aligned prepared data.
    Assumes calculate_stocflexibility() now handles None entries in sols.
    """
    gl_esf = 0.0
    esf_by_state = {}

    for state, prob in y_d.items():
        try:
            data = prepared_data_by_state.get(state)

            if data is None:
                print(f"No valid theta regions for state {state}; skipping state")
                continue

            sols = data.get("filtered_solutions", None)

            if sols is None or len(sols) == 0:
                print(f"No filtered solutions for state {state}; skipping state")
                continue

            if not any(sol is not None for sol in sols):
                print(f"All filtered solutions are None for state {state}; skipping state")
                continue

            sf_idx_gaussian = _safe_sf_call(
                calculate_stocflexibility,
                state=state,
                label="Gaussian",
                sols=sols,
                nq=n_q,
                joint_func=joint_pdf,
                d_vector=d_v,
            )

            if sf_idx_gaussian is not None:
                esf_by_state[state] = sf_idx_gaussian
                gl_esf += sf_idx_gaussian * prob

        except ValueError as e:
            print(f"Skipping state {state} due to ValueError: {e}")
            continue

        print(f"Finished for state {state}.")

    return gl_esf, esf_by_state

In [10]:
def calculate_sm_esf(y_d: dict, prepared_data_by_s: dict, d_v, s_level: int, ns_u=None, ws_u=None):
    """
    Smolyak ESF using stage-aligned prepared data.
    Assumes map_u_to_theta_and_jacobian() now handles None entries in solutions.
    """
    sm_esf = 0.0
    esf_by_state = {}

    for s, p in y_d.items():
        try:
            data = prepared_data_by_s.get(s)

            if data is None:
                print(f"No valid theta regions for state {s}; skipping state")
                continue

            sols = data.get("filtered_solutions", None)

            if sols is None or len(sols) == 0:
                print(f"No filtered solutions for state {s}; skipping state")
                continue

            if not any(sol is not None for sol in sols):
                print(f"All filtered solutions are None for state {s}; skipping state")
                continue

            sf_idx_smolyak = _safe_sf_call(
                calculate_stocflexibility_smolyak,
                state=s,
                label="Smolyak",
                solutions=sols,
                level=s_level,
                joint_func=joint_pdf,
                d_vector=d_v,
                ns_u = ns_u,
                ws_u = ws_u,
            )

            if sf_idx_smolyak is not None:
                esf_by_state[s] = sf_idx_smolyak
                sm_esf += sf_idx_smolyak * p

        except ValueError as e:
            print(f"Skipping state {s} due to ValueError: {e}")
            continue

        print(f"Finished for state {s}.")

    return sm_esf, esf_by_state

In [11]:
def smolyak_esf_with_region_paths(
    solutions,
    dv,
    nsu,
    wsu,
    jointfunc,
    maxidx: int = 0,
    minidx: int = 1,
):
    """
    Python Smolyak ESF that mirrors Gurobi semantics:
    - For each node u_n, try to build a globally consistent region path across stages
      using ppopt's get_region.
    - If any stage fails (no region), node n contributes zero (node_on[n] = 0).
    - Otherwise, contribution is w_n * jac_n * pdf(theta_n).
    """
    n_nodes, n_theta = nsu.shape
    esf = 0.0

    for n in range(n_nodes):
        u_vec = nsu[n, :]

        theta_vals = []
        jacobian = 1.0
        feasible_node = True

        for k, sol in enumerate(solutions):
            # Skip empty/unusable stages as in your existing mapper
            if sol is None:
                theta_k = 0.0
                theta_vals.append(theta_k)
                continue

            # Construct theta_prev,d vector before choosing region
            if isinstance(dv, np.ndarray):
                theta_vector = np.block([np.array(theta_vals, dtype=float), dv])
            else:
                theta_vector = np.array(theta_vals, dtype=float)

            # theta_aug = np.append(theta_vector, 1.0).reshape(-1, 1)

            region = sol.get_region(theta_vector.reshape(-1,1))
            if region is None:
                feasible_node = False
                break
            
            theta_aug = np.append(theta_vector, 1.0).reshape(-1, 1)
            coeffs = np.concatenate((region.A, region.b), axis=1)
            max_coeffs = coeffs[maxidx]
            min_coeffs = coeffs[minidx]

            t_max = (max_coeffs @ theta_aug).item()
            t_min = (min_coeffs @ theta_aug).item()
            length = t_max - t_min

            u_k = float(u_vec[k])
            theta_k = 0.5 * length * u_k + 0.5 * (t_max + t_min)
            theta_vals.append(theta_k)
            jacobian *= 0.5 * length

        if not feasible_node:
            # Mirrors node_on[n] = 0: no contribution from this node
            continue

        theta_arr = np.array(theta_vals, dtype=float)
        esf += jointfunc(theta_arr) * jacobian * wsu[n]

    return esf

In [12]:
def smolyak_esf_region_path_all_states(ydict, prepared_data_by_state, dv, nsu, wsu, jointfunc):
    total_esf = 0.0
    esf_by_state = {}

    for state, prob in ydict.items():
        data = prepared_data_by_state.get(state, None)
        if data is None:
            # no valid theta regions -> skip, contribution = 0
            continue

        sols = data.get("filtered_solutions", None)
        if sols is None or len(sols) == 0 or not any(sol is not None for sol in sols):
            # same skipping logic as calculatesm_esf
            continue

        esf_state = smolyak_esf_with_region_paths(
            solutions=sols,
            dv=dv,
            nsu=nsu,
            wsu=wsu,
            jointfunc=jointfunc,
        )

        esf_by_state[state] = esf_state
        total_esf += prob * esf_state

    return total_esf, esf_by_state

## Smolyak SF Expression

In [13]:
def compute_sf_exprs_regions_smolyak(
    t_bounds_list,
    t_regions_list,
    joint_pdf_expr,
    d_syms,
    theta_syms,
    level: int,
    rule: str = "gaussian",
    growth: bool = True,
):
    """
    FIXED VERSION:
    - For each region combo, contributions from Smolyak nodes are GATED by the region constraints
      after substituting theta(theta_prev, d, u_node).
    - This prevents double-counting across region combos (the main bug in the old version).

    Returns:
      sf_exprs:  list of sympy expressions (each is gated to its combo)
      sf_regions: list of region constraint lists (same as before)
    """
    n_theta = len(theta_syms)

    # Sparse grid nodes/weights on [-1,1]^n (Chaospy gives expectation weights)
    dist = cp.J(*[cp.Uniform(-1, 1) for _ in range(n_theta)])
    ns_u, weights_expectation = cp.generate_quadrature(
        order=level,
        dist=dist,
        rule=rule,
        sparse=True,
        growth=growth,
    )
    ns_u = np.array(ns_u).T
    weights_expectation = np.array(weights_expectation).flatten()

    # Convert expectation weights to integral weights over [-1,1]^n (volume = 2^n)
    ws_u = (2.0 ** n_theta) * weights_expectation

    # Enumerate region combos (same as your current code)
    region_sizes = [bounds.shape[0] for bounds in t_bounds_list]
    region_combos = list(itools.product(*[range(r) for r in region_sizes]))

    sf_exprs = []
    sf_regions = []

    for region_combo in region_combos:

        # 1) Build region constraints in (theta_syms, d_syms)
        combo_constraints = []
        for level_idx, region_idx in enumerate(region_combo):
            rows = t_regions_list[level_idx][region_idx]
            for row in rows:
                t_coeffs = row[:level_idx+1]
                d_coeffs = row[level_idx+1:-1]
                const = row[-1]

                lhs = sum(c * theta_syms[i] for i, c in enumerate(t_coeffs)) + \
                    sum(c * d for c, d in zip(d_coeffs, d_syms)) + const
                ineq = lhs <= 0

                if not isinstance(ineq, (BooleanTrue, BooleanFalse)):
                    combo_constraints.append(ineq)

        # (Optional) stable order; not dedupe, but OK
        combo_constraints = sorted(combo_constraints, key=str)

        # 2) Smolyak quadrature expression with node-wise gating
        sf_sum = 0

        for u_vec, w_u in zip(ns_u, ws_u):

            theta_vals = []
            jacobian = 1

            # Sequentially compute theta_k(u, d) for this combo
            for level_idx, region_idx in enumerate(region_combo):
                bounds = t_bounds_list[level_idx][region_idx]
                bound_inputs = theta_vals + list(d_syms)

                # IMPORTANT: keep your existing convention here:
                # bounds[0] is t_min, bounds[1] is t_max (do NOT change since GL works for you)
                t_min = affine_expr(bounds[0], bound_inputs)
                t_max = affine_expr(bounds[1], bound_inputs)

                u_k = float(u_vec[level_idx])
                t_k = 0.5 * (t_max - t_min) * u_k + 0.5 * (t_max + t_min)

                theta_vals.append(t_k)
                jacobian *= 0.5 * (t_max - t_min)

            # Substitute theta into PDF (so pdf becomes expression in d_syms)
            theta_subs = {sym: val for sym, val in zip(theta_syms, theta_vals)}
            pdf_val = joint_pdf_expr.subs(theta_subs)

            # GATE this node's contribution by combo feasibility
            if combo_constraints:
                gated_constraints = []
                for ineq in combo_constraints:
                    ineq_sub = ineq.subs(theta_subs)
                    # After substitution this should depend only on d_syms (and constants)
                    if not isinstance(ineq_sub, (BooleanTrue, BooleanFalse)):
                        gated_constraints.append(ineq_sub)

                if gated_constraints:
                    cond = sp.And(*gated_constraints)
                    term = sp.Piecewise(
                        (w_u * jacobian * pdf_val, cond),
                        (0, True)
                    )
                else:
                    # constraints evaluated to True/False already
                    term = w_u * jacobian * pdf_val
            else:
                term = w_u * jacobian * pdf_val

            sf_sum += term

        sf_exprs.append(sp.simplify(sf_sum))
        sf_regions.append(combo_constraints)

    return sf_exprs, sf_regions


def compute_sf_smolyak_symbolic_fast(
    t_bounds_list,
    t_regions_list,
    joint_pdf_expr,
    d_syms,
    theta_syms,
    level: int,
    rule: str = "gaussian",
    growth: bool = True,
):
    """
    Smolyak symbolic SF (node-wise region selection) with SAFE Piecewise creation.

    Returns:
        sf_expr (sympy Expr), stats (dict)
    """
    n_theta = len(theta_syms)

    dist = cp.J(*[cp.Uniform(-1, 1) for _ in range(n_theta)])
    ns_u, wE = cp.generate_quadrature(
        order=level, dist=dist, rule=rule, sparse=True, growth=growth
    )
    ns_u = np.asarray(ns_u, dtype=float).T
    wE = np.asarray(wE, dtype=float).ravel()
    wU = (2.0 ** n_theta) * wE

    def _region_holds(k: int, region_rows, theta_prev_exprs):
        """Return sympy Boolean condition (in d_syms only) for stage k region feasibility."""
        conds = []
        for row in region_rows:
            t_coeffs = row[:k]        # theta_0..theta_{k-1}
            d_coeffs = row[k:-1]      # d0..d_{nd-1}
            const = row[-1]

            lhs = sum(c * theta_prev_exprs[i] for i, c in enumerate(t_coeffs)) \
                + sum(c * d for c, d in zip(d_coeffs, d_syms)) \
                + const

            ineq = sp.Le(lhs, 0)
            if not isinstance(ineq, (BooleanTrue, BooleanFalse)):
                conds.append(ineq)

        return sp.And(*conds) if conds else sp.true

    sf_sum = 0

    for u_vec, w_u in zip(ns_u, wU):
        theta_vals = []   # sympy expressions in d_syms
        jac = 1

        for k in range(n_theta):
            bound_inputs = theta_vals + list(d_syms)

            # Build non-nested Piecewise by collecting (expr, cond) pairs
            min_pairs = []
            max_pairs = []

            n_regions_k = t_bounds_list[k].shape[0]
            for r_idx in range(n_regions_k):
                # [tmin_coeffs, tmax_coeffs] (keep your convention)
                bounds = t_bounds_list[k][r_idx]
                region_rows = t_regions_list[k][r_idx]

                cond = _region_holds(k, region_rows, theta_vals)
                tmin_r = affine_expr(bounds[0], bound_inputs)
                tmax_r = affine_expr(bounds[1], bound_inputs)

                min_pairs.append((tmin_r, cond))
                max_pairs.append((tmax_r, cond))

            # IMPORTANT: add a default branch to avoid Sympy as_set/ITE rewrite errors
            # Use the last expression as fallback. (Assumes regions cover the space; if not, it still prevents crashes.)
            pw_tmin = sp.Piecewise(*min_pairs, (min_pairs[-1][0], True))
            pw_tmax = sp.Piecewise(*max_pairs, (max_pairs[-1][0], True))

            u_k = float(u_vec[k])
            t_k = 0.5 * (pw_tmax - pw_tmin) * u_k + 0.5 * (pw_tmax + pw_tmin)

            theta_vals.append(t_k)
            jac *= 0.5 * (pw_tmax - pw_tmin)

        pdf_val = joint_pdf_expr.subs(
            {sym: val for sym, val in zip(theta_syms, theta_vals)})
        sf_sum += w_u * jac * pdf_val

    stats = {"n_nodes": int(ns_u.shape[0])}
    # Avoid simplify() here; it can take forever on Piecewise-heavy expressions
    return sf_sum, stats

## Gaussian Legendre SF Expressions

In [14]:
def compute_sf_exprs_regions(
    t_bounds_list,
    t_regions_list,
    joint_pdf_expr,
    d_syms,
    n_gl_list,
    theta_syms
):
    n_theta = len(theta_syms)
    quad_data = [np.polynomial.legendre.leggauss(n) for n in n_gl_list]
    region_sizes = [bounds.shape[0] for bounds in t_bounds_list]
    region_combos = generate_region_combos(region_sizes, n_gl_list)

    sf_exprs = []
    sf_regions = []

    for region_combo in region_combos:
        combo_ptr = 0
        # Initialize integration paths: (theta_vals, weight, scale, constraints)
        paths = [([], 1, 1, [])]

        for level in range(n_theta):
            xi, wi = quad_data[level]
            new_paths = []

            for theta_vals, weight, scale, constraints in paths:
                region_idx = region_combo[combo_ptr]
                combo_ptr += 1

                bound_inputs = theta_vals + list(d_syms)
                bounds = t_bounds_list[level][region_idx]
                t_min = affine_expr(bounds[0], bound_inputs)
                t_max = affine_expr(bounds[1], bound_inputs)

                # Get level-specific region constraints
                rows = t_regions_list[level][region_idx]
                level_constraints = []
                for row in rows:
                    t_coeffs = row[:level]
                    d_coeffs = row[level:-1]
                    const = row[-1]
                    lhs = sum(c * theta_vals[i] for i, c in enumerate(t_coeffs)) + \
                        sum(c * d for c, d in zip(d_coeffs, d_syms)) + const
                    ineq = lhs <= 0
                    # level_constraints.append(sp.simplify(lhs <= 0))
                    if not isinstance(ineq, (BooleanTrue, BooleanFalse)):
                        level_constraints.append(ineq)

                new_constraints = constraints + level_constraints

                # Quadrature expansion for this level
                for q in range(len(xi)):
                    t = 0.5 * (t_max - t_min) * xi[q] + 0.5 * (t_max + t_min)
                    # new_theta_vals = theta_vals + [sp.simplify(t)]
                    new_theta_vals = theta_vals + [t]
                    new_weight = weight * wi[q]
                    new_scale = scale * 0.5 * (t_max - t_min)
                    new_paths.append(
                        (new_theta_vals, new_weight, new_scale, new_constraints))

            paths = new_paths

        # Final integration and region collection
        sf_sum = 0
        all_constraints = []
        for theta_vals, weight, scale, constraints in paths:
            theta_subs = {sym: val for sym, val in zip(theta_syms, theta_vals)}
            pdf_val = joint_pdf_expr.subs(theta_subs)
            sf_sum += weight * scale * pdf_val
            all_constraints.extend(constraints)

        # Deduplicate constraints symbolically
        unique_constraints = []
        for c in all_constraints:
            if isinstance(c, (BooleanTrue, BooleanFalse)):
                print(f'Skipping trivial constraint: {c}')
            if not any(normalized_lhs(c) == normalized_lhs(u) and type(c) == type(u) for u in unique_constraints if normalized_lhs(u) is not None):
                unique_constraints.append(c)

        sf_exprs.append(sf_sum)
        # sf_regions.append(sorted(all_constraints, key=str))
        # sf_exprs.append(sp.simplify(sf_sum))
        sf_regions.append(sorted(unique_constraints, key=str))

    return sf_exprs, sf_regions

## Gurobipy Model

In [15]:
def build_base_model_esf_gurobi(d_bounds, cost_builder=None, esf_target_init=0.0, budget_limit_init=0.0, objective: str = 'Cost'):
    m = gp.Model("ESF_design")

    d = m.addVars(
        len(d_bounds),
        lb=[b[0] for b in d_bounds],
        ub=[b[1] for b in d_bounds],
        name="d"
    )

    ESF = m.addVar(lb=0, name="ESF")
    Cost = m.addVar(lb=0, name="Cost")

    m._esf_terms = []
    
    if cost_builder is None:
        cost_expr = gp.quicksum(d[j] for j in d.keys())
    else:
        cost_expr = cost_builder(d)
        
    m.addConstr(Cost == cost_expr, name='cost_expr')
    
    if objective=='Cost':
        esf_target_con = m.addConstr(ESF >= esf_target_init, name="esf_target")
        esf_budget_con = None
        m.setObjective(Cost, GRB.MINIMIZE)
        
    elif objective=='ESF':
        esf_budget_con = m.addConstr(Cost <= budget_limit_init, name="budget_limit")
        esf_target_con = None
        m.setObjective(ESF, GRB.MAXIMIZE)
    
    # if esf_target_init is not None:
    #     esf_target_con = m.addConstr(ESF >= esf_target_init, name="esf_target")
    # else:
    #     esf_target_con = None
    # 
    # if budget_limit_init is not None:
    #     esf_budget_con = m.addConstr(Cost <= budget_limit_init, name="budget_limit")
    # else:
    #     esf_budget_con = None
    
    m.update()

    return m, d, ESF, esf_target_con, esf_budget_con

In [16]:
def bound_affine_expr(row, theta_bounds_1d, d_bounds):
    """
    row: 1D array [a_0,...,a_{nθ-1}, b_0,...,b_{nd-1}, c]
    theta_bounds_1d: list of (lb, ub) for the theta entries present in row
    d_bounds: list of (lb, ub) for d_j

    Returns (min_val, max_val) over all feasible (theta, d).
    """
    row = np.asarray(row, dtype=float).ravel()
    n_theta = len(theta_bounds_1d)
    n_d = len(d_bounds)
    assert len(row) == n_theta + n_d + 1

    a = row[:n_theta]
    b = row[n_theta : n_theta + n_d]
    c = row[-1]

    # bound a^T theta
    theta_min = 0.0
    theta_max = 0.0
    for coef, (lb, ub) in zip(a, theta_bounds_1d):
        if coef >= 0:
            theta_min += coef * lb
            theta_max += coef * ub
        else:
            theta_min += coef * ub
            theta_max += coef * lb

    # bound b^T d
    d_min = 0.0
    d_max = 0.0
    for coef, (lb, ub) in zip(b, d_bounds):
        if coef >= 0:
            d_min += coef * lb
            d_max += coef * ub
        else:
            d_min += coef * ub
            d_max += coef * lb

    expr_min = theta_min + d_min + c
    expr_max = theta_max + d_max + c
    return expr_min, expr_max

In [17]:
def compute_bigM_for_state(theta_bounds_list, theta_regions_list, t_bounds, d_bounds):
    """
    Returns:
        M_bound: list over k of array over r with M for tmin/tmax links
        M_region: list over k of array over r with M for region membership
    """
    nt = len(theta_bounds_list)
    nd = len(d_bounds)

    # t_bounds: list of (θ_k_lb, θ_k_ub) for all k
    t_bounds = [tuple(tb) for tb in t_bounds]

    M_bound = []
    M_region = []

    for k in range(nt):
        Rk = theta_bounds_list[k].shape[0]
        M_bound_k = np.zeros(Rk)
        M_region_k = np.zeros(Rk)

        for r in range(Rk):
            # --- 1) tmin/tmax rows ---
            tmin_coeff = theta_bounds_list[k][r][0]
            tmax_coeff = theta_bounds_list[k][r][1]

            # tmin_expr uses first k theta vars (0..k-1) + all d
            theta_bounds_for_row = t_bounds[:k]  # length k

            # bound tmin_expr, tmax_expr
            _, max_tmin = bound_affine_expr(
                tmin_coeff,
                theta_bounds_for_row,
                d_bounds,
            )
            _, max_tmax = bound_affine_expr(
                tmax_coeff,
                theta_bounds_for_row,
                d_bounds,
            )

            # tmin[k,n], tmax[k,n] bounded by t_bounds[k]
            tmin_lb, tmin_ub = t_bounds[k]
            tmax_lb, tmax_ub = t_bounds[k]

            # worst deviation |tmin - tmin_expr|
            max_dev_tmin = max(
                abs(tmin_lb - max_tmin),
                abs(tmin_ub - max_tmin),
            )
            # similarly for tmax
            max_dev_tmax = max(
                abs(tmax_lb - max_tmax),
                abs(tmax_ub - max_tmax),
            )

            M_bound_k[r] = max(max_dev_tmin, max_dev_tmax)

            # --- 2) region membership rows ---
            rows = theta_regions_list[k][r]
            max_violation = 0.0
            for row in rows:
                row = np.asarray(row, dtype=float).ravel()

                # determine how many theta coeffs are in this row (k or k+1)
                theta_count = k
                if len(row) == (k + 1) + nd + 1:
                    theta_count = k + 1
                elif len(row) != theta_count + nd + 1:
                    raise ValueError(
                        f"Bad region row length at k={k}, r={r}. Got {len(row)}."
                    )

                theta_bounds_for_row = t_bounds[:theta_count]

                row_min, row_max = bound_affine_expr(
                    row,
                    theta_bounds_for_row,
                    d_bounds,
                )

                # row <= 0 is active; when z=0 we allow row <= M
                # so M must be >= max possible positive value of row
                max_violation = max(max_violation, max(0.0, row_max))

            M_region_k[r] = max_violation

        M_bound.append(M_bound_k)
        M_region.append(M_region_k)

    return M_bound, M_region

In [18]:
def add_smolyak_sf_block_gurobi(
    m,
    name,
    d_vars,
    t_bounds_list,
    t_regions_list,
    ns_u,
    ws_u,
    pdf_builder,
    prob,
    big_m,
):
    """
    Sparse Smolyak SF block with node_on:
    - builds only valid stages
    - skips states with no valid stages
    - each Smolyak node n has node_on[n] ∈ {0,1}
      * node_on[n] = 1 → normal region selection at all stages
      * node_on[n] = 0 → all z[k,r,n] = 0, jac[n] = 0, zero SF contribution
    - accumulates contribution into m._esf_terms
    """

    if not hasattr(m, "_esf_terms"):
        m._esf_terms = []

    n_theta_total = len(t_bounds_list)
    n_nodes = ns_u.shape[0]
    d_list = [d_vars[j] for j in sorted(d_vars.keys())]

    # stages that have structural data
    valid_stage_indices = [
        k for k in range(n_theta_total)
        if t_bounds_list[k] is not None
        and t_regions_list[k] is not None
        and len(t_regions_list[k]) > 0
    ]

    if not valid_stage_indices:
        print(f"{name}: no valid stages; skipping SF block")
        return {}

    theta = {}
    tmin = {}
    tmax = {}
    scale = {}
    jac = {}
    z = {}
    # stateon = {}
    # node_on = {}
    
    M_bound, M_region = compute_bigM_for_state(
        t_bounds_list, t_regions_list, t_bounds, d_bounds
    )
    
    state_on = m.addVar(vtype=GRB.BINARY, name=f"{name}_state_on")
    
    # Variables per node
    for n in range(n_nodes):
        jac[n] = m.addVar(lb=0.0, name=f"{name}_jac[{n}]")

        for k in valid_stage_indices:
            theta[k, n] = m.addVar(lb=-GRB.INFINITY, name=f"{name}_theta[{k},{n}]")
            tmin[k, n] = m.addVar(lb=-GRB.INFINITY, name=f"{name}_tmin[{k},{n}]")
            tmax[k, n] = m.addVar(lb=-GRB.INFINITY, name=f"{name}_tmax[{k},{n}]")
            scale[k, n] = m.addVar(lb=0.0, name=f"{name}_scale[{k},{n}]")

    # Region selectors
    for k in valid_stage_indices:
        Rk = t_bounds_list[k].shape[0]
        for n in range(n_nodes):
            for r in range(Rk):
                z[k, r, n] = m.addVar(
                    vtype=GRB.BINARY,
                    name=f"{name}_z[{k},{r},{n}]"
                )

    m.update()

    def theta_expr(i, n):
        if (i, n) in theta:
            return theta[i, n]
        return 0.0

    def affine_from_coeffs(coeffs, k, n):
        expr = float(coeffs[-1])

        for i in range(k):
            expr += float(coeffs[i]) * theta_expr(i, n)

        for j, dvar in enumerate(d_list):
            expr += float(coeffs[k + j]) * dvar

        return expr

    # 1) Region selection and region-linked bounds
    for k in valid_stage_indices:
        Rk = t_bounds_list[k].shape[0]

        for n in range(n_nodes):
            # CONDITIONAL region selection:
            # node_on[n] = 1 → sum_r z[k,r,n] = 1
            # node_on[n] = 0 → all z[k,r,n] = 0
            # m.addConstr(
            #     gp.quicksum(z[k, r, n] for r in range(Rk)) == node_on[n],
            #     name=f"{name}_region_select[{k},{n}]"
            # )
            
            m.addConstr(
                gp.quicksum(z[k, r, n] for r in range(Rk)) == state_on,
                name=f"{name}_region_select[{k},{n}]"
            )

            for r in range(Rk):
                bounds = t_bounds_list[k][r]
                tmin_aff = affine_from_coeffs(bounds[0], k, n)
                tmax_aff = affine_from_coeffs(bounds[1], k, n)

                # m.addConstr(
                #     tmin[k, n] >= tmin_aff - big_m * (1 - z[k, r, n]),
                #     name=f"{name}_tmin_lb[{k},{r},{n}]"
                # )
                # m.addConstr(
                #     tmin[k, n] <= tmin_aff + big_m * (1 - z[k, r, n]),
                #     name=f"{name}_tmin_ub[{k},{r},{n}]"
                # )
                # m.addConstr(
                #     tmax[k, n] >= tmax_aff - big_m * (1 - z[k, r, n]),
                #     name=f"{name}_tmax_lb[{k},{r},{n}]"
                # )
                # m.addConstr(
                #     tmax[k, n] <= tmax_aff + big_m * (1 - z[k, r, n]),
                #     name=f"{name}_tmax_ub[{k},{r},{n}]"
                # )
                
                m.addConstr(
                    tmin[k, n] >= tmin_aff - M_bound[k][r] * (1 - z[k, r, n]),
                    name=f"{name}_tmin_lb[{k},{r},{n}]"
                )
                m.addConstr(
                    tmin[k, n] <= tmin_aff + M_bound[k][r] * (1 - z[k, r, n]),
                    name=f"{name}_tmin_ub[{k},{r},{n}]"
                )
                m.addConstr(
                    tmax[k, n] >= tmax_aff - M_bound[k][r] * (1 - z[k, r, n]),
                    name=f"{name}_tmax_lb[{k},{r},{n}]"
                )
                m.addConstr(
                    tmax[k, n] <= tmax_aff + M_bound[k][r] * (1 - z[k, r, n]),
                    name=f"{name}_tmax_ub[{k},{r},{n}]"
                )

                rows = t_regions_list[k][r]
                for row_idx, row in enumerate(rows):
                    lhs = float(row[-1])

                    for i in range(k):
                        lhs += float(row[i]) * theta_expr(i, n)

                    for j, dvar in enumerate(d_list):
                        lhs += float(row[k + j]) * dvar

                    # lhs <= 0 when region active; relaxed by big-M when z=0
                    # m.addConstr(
                    #     lhs <= big_m * (1 - z[k, r, n]),
                    #     name=f"{name}_region_ineq[{k},{r},{n},{row_idx}]"
                    # )
                    
                    m.addConstr(
                        lhs <= M_region[k][r] * (1 - z[k, r, n]),
                        name=f"{name}_region_ineq[{k},{r},{n},{row_idx}]"
                    )

    # 2) Theta mapping and scale only for valid stages
    for n in range(n_nodes):
        u_vec = ns_u[n]

        for k in valid_stage_indices:
            u_k = float(u_vec[k])

            m.addConstr(
                theta[k, n] == 0.5 * (tmax[k, n] - tmin[k, n]) * u_k
                               + 0.5 * (tmax[k, n] + tmin[k, n]),
                name=f"{name}_theta_map[{k},{n}]"
            )

            m.addConstr(
                scale[k, n] == 0.5 * (tmax[k, n] - tmin[k, n]),
                name=f"{name}_scale_def[{k},{n}]"
            )

    # 3) Jacobian chain only across valid stages
    jac_chain = {}
    for n in range(n_nodes):
        if len(valid_stage_indices) == 1:
            k0 = valid_stage_indices[0]
            m.addConstr(jac[n] == scale[k0, n], name=f"{name}_jac_def[{n}]")
        else:
            k0 = valid_stage_indices[0]
            prev = scale[k0, n]

            for step_idx, k in enumerate(valid_stage_indices[1:], start=1):
                jac_chain[step_idx, n] = m.addVar(
                    lb=0.0, name=f"{name}_jac_step[{step_idx},{n}]"
                )
                m.addConstr(
                    jac_chain[step_idx, n] == prev * scale[k, n],
                    name=f"{name}_jac_chain[{step_idx},{n}]"
                )
                prev = jac_chain[step_idx, n]

            m.addConstr(jac[n] == prev, name=f"{name}_jac_def[{n}]")

    # 3b) Force jac[n] to zero when node_on[n] = 0
    M_jac = 1e5  # TODO: tighten if you have a better bound
    for n in range(n_nodes):
        # m.addConstr(
        #     jac[n] <= M_jac * node_on[n],
        #     name=f"{name}_jac_off[{n}]"
        # )
        
        m.addConstr(
            jac[n] <= M_jac * state_on,
            name=f"{name}_jac_off[{n}]"
        )

    # 4) Ensure required theta indices for the pdf exist
    required_pdf_stages = [0, 1]  # theta[0,n] = S, theta[1,n] = D
    if any(k not in valid_stage_indices for k in required_pdf_stages):
        print(f"{name}: required pdf theta stage missing; skipping SF block")
        return {}

    node_sf = {}
    prod = {}

    for n in range(n_nodes):
        prod[n] = m.addVar(lb=-GRB.INFINITY, name=f"{name}_prod[{n}]")
        node_sf[n] = m.addVar(lb=-GRB.INFINITY, name=f"{name}_node_sf[{n}]")

        pdf_expr = pdf_builder(theta, n)

        m.addConstr(
            prod[n] == jac[n] * pdf_expr,
            name=f"{name}_prod_def[{n}]"
        )

        m.addConstr(
            node_sf[n] == float(ws_u[n]) * prod[n],
            name=f"{name}_node_sf_def[{n}]"
        )

    # 5) Block-level SF and ESF accumulation
    sf_block = m.addVar(lb=0.0, name=f"{name}_sf")
    m.addConstr(
        sf_block == gp.quicksum(node_sf[n] for n in range(n_nodes)),
        name=f"{name}_sf_def"
    )

    m._esf_terms.append(prob * sf_block)

    return {
        "name": name,
        "valid_stage_indices": valid_stage_indices,
        "theta": theta,
        "tmin": tmin,
        "tmax": tmax,
        "scale": scale,
        "jac": jac,
        "z": z,
        # "node_on": node_on,
        "state_on": state_on,
        "prod": prod,
        "node_sf": node_sf,
        "sf_block": sf_block,
        "t_bounds_list": t_bounds_list,
    }

In [19]:
def add_all_smolyak_blocks_for_optimization(
    m,
    d_vars,
    y_d,
    prepared_data_by_state,
    ns_u,
    ws_u,
    pdf_builder,
    big_m=1e4,
):
    blk_map = {}
    for state, prob in y_d.items():
        data = prepared_data_by_state.get(state)

        if data is None:
            print(f"[Gurobi] No prepared data for state {state}; SF contribution = 0.")
            continue

        t_bounds_list = data["filtered_theta_bounds"]
        t_regions_list = data["filtered_theta_regions"]

        has_any_valid_stage = any(
            (tb is not None) and (tr is not None and len(tr) > 0)
            for tb, tr in zip(t_bounds_list, t_regions_list)
        )
        if not has_any_valid_stage:
            print(f"[Gurobi] State {state} has no valid stages after filtering; SF=0, no block.")
            continue

        print(f"[Gurobi] Adding SF block for state {state} with prob {prob}.")
        block = add_smolyak_sf_block_gurobi(
            m=m,
            name=f"sf_{state}",
            d_vars=d_vars,
            t_bounds_list=t_bounds_list,
            t_regions_list=t_regions_list,
            ns_u=ns_u,
            ws_u=ws_u,
            pdf_builder=pdf_builder,
            prob=prob,
            big_m=big_m,
        )
        blk_map[state] = block

    return blk_map

In [20]:
def smolyak_nodes_weights(
    n_theta: int,
    level: int,
    rule: str = "gaussian",
    growth: bool = True,
):
    """
    Returns:
        ns_u: (N, n_theta) array of Smolyak nodes in u-space (each u_k in [-1,1])
        ws_u: (N,) array of weights for integrating over [-1,1]^n_theta
                   i.e., sum_i ws_u[i] * f(ns_u[i]) ≈ ∫_{[-1,1]^n} f(u) du
    """
    dist = cp.J(*[cp.Uniform(-1, 1) for _ in range(n_theta)])

    # Chaospy returns nodes shape (n_theta, N) and weights for expectation
    nodes, wE = cp.generate_quadrature(
        order=level,
        dist=dist,
        rule=rule,
        sparse=True,
        growth=growth,
    )

    nodes = np.asarray(nodes, dtype=float)        # (n_theta, N)
    wE = np.asarray(wE, dtype=float).ravel()      # (N,)

    ns_u = nodes.T                             # (N, n_theta)

    # Convert expectation weights to integral weights over [-1,1]^n:
    # E[f(U)] = ∫ f(u) p(u) du with p(u)=1/2^n on [-1,1]^n
    # => ∫ f(u) du = 2^n * E[f(U)]
    ws_u = (2.0 ** n_theta) * wE

    return ns_u, ws_u

## Diagnostics

In [21]:
def debug_python_smolyak_nodes_for_state(state, dv, nsu, wsu):
    """
    For a single state, compute Smolyak per-node contributions in Python.

    Returns: list of dicts with keys:
        'n', 'u', 'theta', 'jac', 'w', 'pdf', 'contrib', 'feasible'
    """
    data = prepared_data_by_state.get(state)
    if data is None:
        raise ValueError(f"No prepared data for state {state}")

    sols = data.get("filtered_solutions")
    if sols is None or len(sols) == 0 or not any(sol is not None for sol in sols):
        raise ValueError(f"No valid filtered solutions for state {state}")

    results = []
    dv = np.array(dv, dtype=float)

    n_nodes, _ = nsu.shape
    for n in range(n_nodes):
        u_vec = nsu[n, :]
        try:
            theta_vals, jacobian = map_u_to_theta_and_jacobian(
                solutions=sols,
                u=u_vec,
                dvector=dv,
            )
            # jointpdf is your joint PDF over theta for this example
            pdf_val = joint_pdf(theta_vals)
            contrib = pdf_val * jacobian * wsu[n]
            feasible = True
        except ValueError as e:
            # e.g., "not inside any critical region"
            theta_vals = None
            jacobian = 0.0
            pdf_val = 0.0
            contrib = 0.0
            feasible = False

        results.append({
            "n": n,
            "u": u_vec.copy(),
            "theta": theta_vals,
            "jac": float(jacobian),
            "w": float(wsu[n]),
            "pdf": float(pdf_val),
            "contrib": float(contrib),
            "feasible": feasible,
        })

    return results

In [22]:
def debug_gurobi_smolyak_nodes_for_state(fixed_m, block_info, st, nsu, wsu, state_label="S_state"):
    """
    After optimizing the model, extract per-node data from the Gurobi Smolyak block.

    block_info: dict returned by addsmolyaksfblockgurobi for this state.
    Returns: list of dicts, same shape as Python debug.
    """
    theta = block_info[st]["theta"]      # dict (k, n) -> Var
    jac = block_info[st]["jac"]          # dict n -> Var
    nodeon = block_info[st]["node_on"]    # dict n -> Var
    nodesf = block_info[st]["node_sf"]    # dict n -> Var

    n_nodes, _ = nsu.shape
    results = []

    for n in range(n_nodes):
        # node_on_val = float(nodeon[n].X)
        # jac_val = float(jac[n].X)
        # nodesf_val = float(nodesf[n].X)
        # node_on_val = fixed_m.getVarByName(f"sf_{st}_node_on[{n}]").X
        state_val = fixed_m.getVarByName(f"sf_{st}_state_on").X
        jac_val = fixed_m.getVarByName(f"sf_{st}_jac[{n}]").X
        nodesf_val = fixed_m.getVarByName(f"sf_{st}_node_sf[{n}]").X

        theta_vals = []
        k_list = sorted({k for (k_idx, n_idx) in theta.keys() if n_idx == n for k in [k_idx]})
        for k in k_list:
            # theta_vals.append(theta[(k, n)].X)
            theta_vals.append(fixed_m.getVarByName(f"sf_{st}_theta[({k,n})]"))
        theta_vals = np.array(theta_vals, dtype=float) if theta_vals else None

        # We do not recompute pdf here; nodesf already equals w_u[n] * jac[n] * pdf(theta_n)
        contrib = nodesf_val
        u_vec = nsu[n, :]

        results.append({
            "n": n,
            "u": u_vec.copy(),
            "theta": theta_vals,
            "jac": jac_val,
            "w": float(wsu[n]),
            "state_on": state_val,
            # "node_on": node_on_val,
            "contrib": contrib,
        })

    return results

In [23]:
def compare_py_vs_gurobi_nodes(py_nodes, grb_nodes, tol_jac=1e-6, tol_theta=1e-6, tol_contrib=1e-6):
    """
    Compare Python and Gurobi per-node data, assuming same node ordering.
    Prints mismatches and returns a summary dict.
    """
    assert len(py_nodes) == len(grb_nodes)
    n_nodes = len(py_nodes)

    mismatches = {
        "jac": [],
        "theta": [],
        "active_diff": [],
        "contrib": [],
    }

    esf_py = sum(item["contrib"] for item in py_nodes)
    esf_grb = sum(item["contrib"] for item in grb_nodes if item["node_on"] > 0.5)

    print(f"Total ESF (Python): {esf_py:.12f}")
    print(f"Total ESF (Gurobi): {esf_grb:.12f}")
    print(f"Absolute difference: {abs(esf_py - esf_grb):.12f}")

    for n in range(n_nodes):
        p = py_nodes[n]
        g = grb_nodes[n]

        # Activation mismatch: Python feasible vs Gurobi node_on
        py_active = p["feasible"]
        grb_active = g["node_on"] > 0.5
        if py_active != grb_active:
            mismatches["active_diff"].append(n)
            print(f"[Node {n}] activation mismatch: Python feasible={py_active}, Gurobi node_on={grb_active}")

        # Compare jacobian when both active
        if py_active and grb_active:
            if abs(p["jac"] - g["jac"]) > tol_jac:
                mismatches["jac"].append(n)
                print(f"[Node {n}] jac mismatch: py={p['jac']:.10e}, grb={g['jac']:.10e}")

            if p["theta"] is not None and g["theta"] is not None:
                if len(p["theta"]) == len(g["theta"]):
                    theta_diff = np.max(np.abs(p["theta"] - g["theta"]))
                    if theta_diff > tol_theta:
                        mismatches["theta"].append(n)
                        print(f"[Node {n}] theta mismatch (max abs diff): {theta_diff:.10e}")
                else:
                    mismatches["theta"].append(n)
                    print(f"[Node {n}] theta length mismatch: py={len(p['theta'])}, grb={len(g['theta'])}")

            # Compare contributions
            if abs(p["contrib"] - g["contrib"]) > tol_contrib:
                mismatches["contrib"].append(n)
                print(f"[Node {n}] contrib mismatch: py={p['contrib']:.10e}, grb={g['contrib']:.10e}")

    return {
        "esf_py": esf_py,
        "esf_grb": esf_grb,
        "mismatches": mismatches,
    }

## Computational Output

In [24]:
def make_gap_callback_with_label(run_label, store_every=5.0):
    history = []

    def cb(model, where):
        if where == GRB.Callback.MIP:
            runtime = model.cbGet(GRB.Callback.RUNTIME)
            if runtime - model._last_record_time < store_every:
                return
            model._last_record_time = runtime

            objbst = model.cbGet(GRB.Callback.MIP_OBJBST)
            objbnd = model.cbGet(GRB.Callback.MIP_OBJBND)

            gap = None
            if abs(objbst) < GRB.INFINITY and abs(objbnd) < GRB.INFINITY:
                gap = abs(objbst - objbnd) / max(1e-10, abs(objbst))

            history.append({
                "run": run_label,
                "runtime_sec": float(runtime),
                "nodecnt": float(model.cbGet(GRB.Callback.MIP_NODCNT)),
                "itcnt": float(model.cbGet(GRB.Callback.MIP_ITRCNT)),
                "obj_best": None if abs(objbst) >= GRB.INFINITY else float(objbst),
                "obj_bound": None if abs(objbnd) >= GRB.INFINITY else float(objbnd),
                "gap_pct": None if gap is None else 100.0 * gap,
            })

    return cb, history

## Case Study

In [25]:
# # Bansal (2000) Illustrative Example
# y_dict = {
#     (0.5,0.5): 0.01,
#     (0.5,1): 0.09,
#     (1,0.5): 0.09,
#     (1,1): 0.81,
# }
# 
# feasibility_algo = mpqp_algorithm.geometric_parallel
# theta_bounds_algo = mpqp_algorithm.geometric_parallel
# 
# feas_algo_name = 'gp' if feasibility_algo.name in ['geometric_parallel', 'geometric'] else ('cb' if feasibility_algo in ['combinatorial', 'combinatorial_parallel'] else '')
# tb_algo_name = 'gp' if theta_bounds_algo.name in ['geometric_parallel', 'geometric'] else ('cb' if feasibility_algo in ['combinatorial', 'combinatorial_parallel'] else '')
# 
# t_bounds = [(0, 4), (0, 4)]
# d_bounds = [(0, 5), (0, 5)]
# nt = len(t_bounds)
# nd = len(d_bounds)
# 
# def create_flexibility_model(tbounds:list, dbounds:list, y_list: tuple = None):
#     m = MPModeler()
# 
#     u = m.add_var(name='u')
#     x = m.add_var(name='x')
#     z = m.add_var(name='z')
# 
#     t1 = m.add_param(name='t1')
#     t2 = m.add_param(name='t2')
#     d1 = m.add_param(name='d1')
#     d2 = m.add_param(name='d2')
#     m.add_constr(2*x - 3*z + t1 - d2 == 0)
#     m.add_constr(x - z/2 - t1/2 + t2/2 + d1 - 7*d2/2 <= u)
#     m.add_constr(-2*x + 2*z - 4*t1/3 - t2 + 2*d2 + 1/3 <= u)
#     m.add_constr(-x + 5*z/2 + t1/2 - t2 - d1 + d2/2 - 1 <= u)
#     m.add_constr(-50 <= x)
#     m.add_constr(-50 <= z)
#     m.add_constr(tbounds[0][0] <= t1)
#     m.add_constr(tbounds[1][0] <= t2)
#     m.add_constr(dbounds[0][0] <= d1)
#     m.add_constr(dbounds[1][0] <= d2)
#     m.add_constr(t1 <= tbounds[0][1])
#     m.add_constr(t2 <= tbounds[1][1])
#     m.add_constr(d1 <= dbounds[0][1] * y_list[0])
#     m.add_constr(d2 <= dbounds[1][1] * y_list[1])
#     m.set_objective(u)
# 
#     return m
# 
# def joint_pdf(theta: list):
#     return (2/np.pi)*np.exp(-2*((theta[0]-2)**2 + (theta[1]-2)**2))
# 
# def pdf_builder(theta, n, eps=1e-6):
#     theta0 = theta[(0, n)]
#     theta1 = theta[(1, n)]
#     return (2/math.pi) * nlfunc.exp(-2 * ((theta0-2) * (theta0-2) + (theta1-2) * (theta1-2)))

In [26]:
# Bansal (2000) Process Example 1
y_dict = {
    (0, 0, 0): 0.001,
    (0, 0, 1): 0.003,
    (0, 1, 0): 0.006,
    (1, 0, 0): 0.010,
    (0, 1, 1): 0.040,
    (1, 0, 1): 0.066,
    (1, 1, 0): 0.114,
    (1, 1, 1): 0.760
}

t_bounds = [(8, 16), (3, 11)]
d_bounds = [(0, 10), (0, 10), (0, 10)]
nt = len(t_bounds)
nd = len(d_bounds)

feasibility_algo = mpqp_algorithm.geometric_parallel
theta_bounds_algo = mpqp_algorithm.geometric_parallel

feas_algo_name = 'gp' if feasibility_algo.name in ['geometric_parallel', 'geometric'] else ('cb' if feasibility_algo in ['combinatorial', 'combinatorial_parallel'] else '')
tb_algo_name = 'gp' if theta_bounds_algo.name in ['geometric_parallel', 'geometric'] else ('cb' if feasibility_algo in ['combinatorial', 'combinatorial_parallel'] else '')

def create_flexibility_model(tbounds: list, dbounds: list, y_list: tuple = None):
    j1 = 0.92
    j2 = 0.85
    j3 = 0.75

    m = MPModeler()

    u = m.add_var(name='u')
    F1 = m.add_var(name="F1")
    F2 = m.add_var(name="F2")
    F3 = m.add_var(name="F3")
    F4 = m.add_var(name="F4")
    F5 = m.add_var(name="F5")
    F6 = m.add_var(name="F6")
    F7 = m.add_var(name="F7")

    S = m.add_param(name='S')
    D = m.add_param(name='D')

    d1 = m.add_param(name='d1')
    d2 = m.add_param(name='d2')
    d3 = m.add_param(name='d3')

    m.add_constr(F4 - j1 * F2 == 0)
    m.add_constr(F1 - F2 - F3 == 0)
    m.add_constr(F5 - j2 * F4 == 0)
    m.add_constr(F6 - j3 * F3 == 0)
    m.add_constr(F7 - F5 - F6 == 0)
    m.add_constr(F1 - S <= u)
    m.add_constr(D - F7 <= u)
    m.add_constr(F2 - d1 * y_list[0] <= u)
    m.add_constr(F4 - d2 * y_list[1] <= u)
    m.add_constr(F3 - d3 * y_list[2] <= u)

    # for v in [F1, F2, F3, F4, F5, F6, F7]:
    #     m.add_constr(v >= 0)

    m.add_constr(tbounds[0][0] + 1e-6 <= S)
    m.add_constr(S <= tbounds[0][1])
    m.add_constr(tbounds[1][0] + 1e-6 <= D)
    m.add_constr(D <= tbounds[1][1])

    m.add_constr(dbounds[0][0] <= d1)
    m.add_constr(d1 <= dbounds[0][1])
    m.add_constr(dbounds[1][0] <= d2)
    m.add_constr(d2 <= dbounds[1][1])
    m.add_constr(dbounds[2][0] <= d3)
    m.add_constr(d3 <= dbounds[2][1])

    m.set_objective(u)

    return m

def joint_pdf(theta: list):
    Sval, Dval = theta
    eps = 1e-12
    x = max(Sval - 8.0, eps)
    return (1 / (1.2 * np.pi * x)) * np.exp(
        -1.39 * (np.log(x)) ** 2 - 0.5 * (Dval - 7.0) ** 2)

def cost_function(d):
    coeffs = [2,3,5]
    return gp.quicksum(coeffs[j] * d[j] for j in d.keys())

def pdf_builder(theta, n, eps=1e-6):
    S = theta[(0, n)]
    D = theta[(1, n)]
    x = S - 8.0

    return (
        1.0 / (1.2 * math.pi)
        * (1.0 / (x + eps))
        * nlfunc.exp(
            -1.39 * nlfunc.log(x + eps) * nlfunc.log(x + eps)
            - 0.5 * (D - 7.0) * (D - 7.0)
        )
    )

In [27]:
prepared_data_by_state = {
    state: _prepare_state_data(
        state,
        tbounds=t_bounds,
        dbounds=d_bounds,
        solve_algo=feasibility_algo,
        theta_algo=theta_bounds_algo,
    )
    for state in y_dict
}

load_prepared_data_by_state = prepared_data_by_state

Set parameter Username
Academic license - for non-commercial use only - expires 2027-02-12
Using a found active set [0, 1, 2, 3, 4, 6, 7, 8]
Spawned threads across 24
 Number of Facets to look at this time 10
[theta 0] MPLP infeasible / zero Chebyshev ball: The chebychev ball has either a radius of zero, or the problem is not feasible!
Finished solving for theta1
[theta 1] MPLP infeasible / zero Chebyshev ball: The chebychev ball has either a radius of zero, or the problem is not feasible!
Finished solving for theta2
Using a found active set [0, 1, 2, 3, 4, 6, 7, 9]
Spawned threads across 24
 Number of Facets to look at this time 12
 Number of Facets to look at this time 19
Using a found active set [6, 7, 8, 9]
Spawned threads across 24
 Number of Facets to look at this time 6
Finished solving for theta1
Using a found active set [0, 6]
Spawned threads across 24
 Number of Facets to look at this time 8
 Number of Facets to look at this time 7
Finished solving for theta2
Using a found ac

In [28]:
# with open(f'pkl_output\prepared_data_by_state_new.pkl', 'wb') as f:
#     pickle.dump(prepared_data_by_state,f)

In [29]:
# with open(f'pkl_output\prepared_data_by_state_new.pkl', 'rb') as f:
#     load_prepared_data_by_state = pickle.load(f)

In [30]:
quad_method = 'smolyak'
smolyak_level = 4
MIP_gap = 0.02

In [31]:
nodes_u, weights_u = smolyak_nodes_weights(n_theta=len(t_bounds), level=smolyak_level, rule="gaussian", growth=True)

esf_gurobi_model, d_vars, ESF, esf_target_con, esf_budget_con = build_base_model_esf_gurobi(d_bounds=d_bounds, cost_builder=cost_function)

In [32]:
block_map = add_all_smolyak_blocks_for_optimization(
    m = esf_gurobi_model,
    d_vars = d_vars,
    y_d = y_dict,
    prepared_data_by_state = load_prepared_data_by_state,
    ns_u = nodes_u,
    ws_u = weights_u,
    pdf_builder = pdf_builder,
)

[Gurobi] State (0, 0, 0) has no valid stages after filtering; SF=0, no block.
[Gurobi] Adding SF block for state (0, 0, 1) with prob 0.003.
[Gurobi] State (0, 1, 0) has no valid stages after filtering; SF=0, no block.
[Gurobi] State (1, 0, 0) has no valid stages after filtering; SF=0, no block.
[Gurobi] Adding SF block for state (0, 1, 1) with prob 0.04.
[Gurobi] Adding SF block for state (1, 0, 1) with prob 0.066.
[Gurobi] Adding SF block for state (1, 1, 0) with prob 0.114.
[Gurobi] Adding SF block for state (1, 1, 1) with prob 0.76.
Warning for adding constraints: zero or small (< 1e-13) coefficients, ignored


In [33]:
ESF = esf_gurobi_model.getVarByName("ESF")
# esf_gurobi_model.addConstr(ESF == esf_gurobi_model._ESF_expr, name="esf_def")
esf_gurobi_model.addConstr(ESF == gp.quicksum(esf_gurobi_model._esf_terms), name="esf_def")
esf_gurobi_model.update()

In [44]:
esf_def_con = esf_gurobi_model.getConstrByName("esf_def")

In [47]:
print(f"{esf_gurobi_model.getRow(esf_def_con)} {esf_def_con.Sense} {esf_def_con.RHS}")

ESF + -0.003 sf_(0, 0, 1)_sf + -0.04 sf_(0, 1, 1)_sf + -0.066 sf_(1, 0, 1)_sf + -0.114 sf_(1, 1, 0)_sf + -0.76 sf_(1, 1, 1)_sf = 0.0


In [48]:
sf_def_con = esf_gurobi_model.getConstrByName(f"sf_(1, 1, 1)_sf_def")

In [49]:
print(f"{esf_gurobi_model.getRow(sf_def_con)} {sf_def_con.Sense} {sf_def_con.RHS}")

-1.0 sf_(1, 1, 1)_node_sf[0] + -1.0 sf_(1, 1, 1)_node_sf[1] + -1.0 sf_(1, 1, 1)_node_sf[2] + -1.0 sf_(1, 1, 1)_node_sf[3] + -1.0 sf_(1, 1, 1)_node_sf[4] + -1.0 sf_(1, 1, 1)_node_sf[5] + -1.0 sf_(1, 1, 1)_node_sf[6] + -1.0 sf_(1, 1, 1)_node_sf[7] + -1.0 sf_(1, 1, 1)_node_sf[8] + -1.0 sf_(1, 1, 1)_node_sf[9] + -1.0 sf_(1, 1, 1)_node_sf[10] + -1.0 sf_(1, 1, 1)_node_sf[11] + -1.0 sf_(1, 1, 1)_node_sf[12] + -1.0 sf_(1, 1, 1)_node_sf[13] + -1.0 sf_(1, 1, 1)_node_sf[14] + -1.0 sf_(1, 1, 1)_node_sf[15] + -1.0 sf_(1, 1, 1)_node_sf[16] + -1.0 sf_(1, 1, 1)_node_sf[17] + -1.0 sf_(1, 1, 1)_node_sf[18] + -1.0 sf_(1, 1, 1)_node_sf[19] + -1.0 sf_(1, 1, 1)_node_sf[20] + -1.0 sf_(1, 1, 1)_node_sf[21] + -1.0 sf_(1, 1, 1)_node_sf[22] + -1.0 sf_(1, 1, 1)_node_sf[23] + -1.0 sf_(1, 1, 1)_node_sf[24] + -1.0 sf_(1, 1, 1)_node_sf[25] + -1.0 sf_(1, 1, 1)_node_sf[26] + -1.0 sf_(1, 1, 1)_node_sf[27] + -1.0 sf_(1, 1, 1)_node_sf[28] + -1.0 sf_(1, 1, 1)_node_sf[29] + -1.0 sf_(1, 1, 1)_node_sf[30] + -1.0 sf_(1, 1, 1)_

In [52]:
sf_node_def_con = esf_gurobi_model.getConstrByName("sf_(1, 1, 1)_node_sf_def[0]")

In [53]:
print(f"{esf_gurobi_model.getRow(sf_node_def_con)} {sf_node_def_con.Sense} {sf_node_def_con.RHS}")

-0.4738537701123776 sf_(1, 1, 1)_prod[0] + sf_(1, 1, 1)_node_sf[0] = 0.0


In [43]:
esf_gurobi_model.write("esf_gurobi_model.rlp")

In [36]:
esf_def_cons = esf_gurobi_model.getGenConstrs()

In [41]:
esf_def_cons

AttributeError: 'gurobipy.GenConstr' object has no attribute 'RHS'

In [34]:
esf_gurobi_model.Params.FuncNonlinear = 1 
esf_gurobi_model.Params.MIPGap = MIP_gap
esf_gurobi_model.Params.NonConvex = 2
esf_gurobi_model.Params.MIPFocus = 1
esf_gurobi_model.Params.Presolve = 2
esf_gurobi_model.Params.Heuristics = 0.5
esf_gurobi_model.Params.NoRelHeurTime = 60
# esf_gurobi_model.Params.DualReductions = 1
esf_gurobi_model.Params.MIPFocus = 3   # or 3 to focus heavily on bound
esf_gurobi_model.Params.Cuts = 2
esf_gurobi_model.Params.TimeLimit=1200
esf_gurobi_model.update()

Set parameter FuncNonlinear to value 1
Set parameter MIPGap to value 0.02
Set parameter NonConvex to value 2
Set parameter MIPFocus to value 1
Set parameter Presolve to value 2
Set parameter Heuristics to value 0.5
Set parameter NoRelHeurTime to value 60
Set parameter MIPFocus to value 3
Set parameter Cuts to value 2
Set parameter TimeLimit to value 1200


## ESF targeted optimization

In [36]:
esf_target = 0.05
esf_target_con.RHS = esf_target
esf_gurobi_model.update()

### Optional Start

In [88]:
# fixed_d = [0, 0, 0] # ESF = 0.00
fixed_d = [6.87,6.35,0] # ESF = 0.05

m_fix = esf_gurobi_model.copy()
for j, val in enumerate(fixed_d):
    v = m_fix.getVarByName(f"d[{j}]")
    v.lb = val
    v.ub = val

# # m_fix.Params.DualReductions = 0

In [89]:
esf_init, sm_esf_by_state_init = calculate_sm_esf(y_dict, load_prepared_data_by_state, np.array(fixed_d), s_level=smolyak_level, ns_u=nodes_u, ws_u=weights_u)

All filtered solutions are None for state (0, 0, 0); skipping state
Skipping Smolyak SF for state (0, 0, 1): The provided theta_vector is not inside any critical region of the solution.
Finished for state (0, 0, 1).
All filtered solutions are None for state (0, 1, 0); skipping state
All filtered solutions are None for state (1, 0, 0); skipping state
Skipping Smolyak SF for state (0, 1, 1): The provided theta_vector is not inside any critical region of the solution.
Finished for state (0, 1, 1).
Skipping Smolyak SF for state (1, 0, 1): The provided theta_vector is not inside any critical region of the solution.
Finished for state (1, 0, 1).
Smolyak stochastic flexibility computed in 0.0058 seconds.
Finished for state (1, 1, 0).
Smolyak stochastic flexibility computed in 0.0049 seconds.
Finished for state (1, 1, 1).


In [90]:
esf_init

0.05036865112748945

In [91]:
sm_esf_by_state_init

{(1, 1, 0): 0.057630035614976385, (1, 1, 1): 0.0576300356149765}

In [92]:
states_to_fix = [s for s in sm_esf_by_state_init if sm_esf_by_state_init[s] != 0]

In [93]:
states_to_fix

[(1, 1, 0), (1, 1, 1)]

In [94]:
for s in states_to_fix:
    temp_var = m_fix.getVarByName(f"sf_{s}_state_on")
    temp_var.lb = 1
    temp_var.ub = 1

In [95]:
m_fix.update()

In [96]:
m_fix.optimize()

Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 13th Gen Intel(R) Core(TM) i7-13700, instruction set [SSE2|AVX|AVX2]
Thread count: 16 physical cores, 24 logical processors, using up to 24 threads

Non-default parameters:
TimeLimit  1200
MIPGap  0.02
Heuristics  0.5
MIPFocus  3
NoRelHeurTime  60
Cuts  2
NonConvex  2
Presolve  2

Optimize a model with 15133 rows, 4250 columns and 36811 nonzeros
Model fingerprint: 0x7381a221
Model has 275 quadratic constraints
Model has 275 general nonlinear constraints (1925 nonlinear terms)
Variable types: 3310 continuous, 940 integer (940 binary)
Coefficient statistics:
  Matrix range     [3e-03, 1e+05]
  QMatrix range    [1e+00, 1e+00]
  QLMatrix range   [1e+00, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 7e+00]
  RHS range        [9e-16, 3e+01]
Presolve removed 14022 rows and 2590 columns
Presolve time: 0.04s
Presolved: 1111 rows, 1660 columns, 2880 nonzeros
Variable types: 1322 c

In [97]:
d0_fix = m_fix.getVarByName(f"d[{0}]")
print(f'd0: {d0_fix.X}')

d1_fix = m_fix.getVarByName(f"d[{1}]")
print(f'd1: {d1_fix.X}')

d2_fix = m_fix.getVarByName(f"d[{2}]")
print(f'd2: {d2_fix.X}')

ESF_fix = m_fix.getVarByName(f"ESF")
print(f'ESF: {ESF_fix.X}')

d0: 6.87
d1: 6.35
d2: 0.0
ESF: 0.05036869710187255


In [98]:
start_values = {v.VarName: v.X for v in m_fix.getVars()}

In [99]:
for v in esf_gurobi_model.getVars():
    if v.VarName in start_values:
        v.Start = start_values[v.VarName]

esf_gurobi_model.update()

In [100]:
# esf_by_state_grb = {}
# for state, prob in y_dict.items():
#     v = m_fix.getVarByName(f"sf_{state}_sf")  # adapt name to your scheme
#     if v is not None:
#         esf_by_state_grb[state] = v.X
# 
# print("Gurobi per-state ESF:", esf_by_state_grb)

In [101]:
# m_test.computeIIS()
# m_test.write("node_infeasible.ilp")

In [102]:
# print("ESF =", m_fix.getVarByName("ESF").X)

In [103]:
# esf_gurobi_model.setObjective(0.0, GRB.MINIMIZE)

## Optimization Start

In [104]:
# if esf_target and esf_target != 0.05:
#     with open(f"pkl_output\MIPGap_{int(MIP_gap*100):03d}\\results_{int((esf_target-0.05)*100):03d}_{quad_method}_{smolyak_level}.pkl", 'rb') as file:
#         load_prev_results = pickle.load(file)
#         print(load_prev_results)
# 
#     for j in d_vars:
#         print(load_prev_results[f'd{j}'])
#         d_vars[j].lb = load_prev_results[f'd{j}']
# 
#     esf_gurobi_model.update()
# 
# elif esf_target == 0.05:
#     d_v_lb = [0.095, 0.095, 6.95]
#     for i, v in enumerate(d_v_lb):
#         d_vars[i].lb = v
# 
# esf_gurobi_model.update()

In [105]:
print(d_vars[0].lb)
print(d_vars[1].lb)
print(d_vars[2].lb)

0.0
0.0
0.0


In [106]:
print(d_vars[0].ub)
print(d_vars[1].ub)
print(d_vars[2].ub)

10.0
10.0
10.0


In [107]:
esf_gurobi_model._last_record_time = -1e100
cb, hist_target = make_gap_callback_with_label(f"target_{esf_target}")

In [108]:
esf_gurobi_model.optimize(cb)

Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 13th Gen Intel(R) Core(TM) i7-13700, instruction set [SSE2|AVX|AVX2]
Thread count: 16 physical cores, 24 logical processors, using up to 24 threads

Non-default parameters:
TimeLimit  1200
MIPGap  0.02
Heuristics  0.5
MIPFocus  3
NoRelHeurTime  60
Cuts  2
NonConvex  2
Presolve  2

Optimize a model with 15133 rows, 4250 columns and 36811 nonzeros
Model fingerprint: 0xcc98ec58
Model has 275 quadratic constraints
Model has 275 general nonlinear constraints (1925 nonlinear terms)
Variable types: 3310 continuous, 940 integer (940 binary)
Coefficient statistics:
  Matrix range     [3e-03, 1e+05]
  QMatrix range    [1e+00, 1e+00]
  QLMatrix range   [1e+00, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+01]
  RHS range        [9e-16, 3e+01]

Loaded user MIP start with objective 32.79
MIP start from previous solve did not produce a new incumbent solution
Presolve model has 275 

In [ ]:
esf_by_state_grb = {}
for state, prob in y_dict.items():
    v = esf_gurobi_model.getVarByName(f"sf_{state}_sf")  # adapt name to your scheme
    if v is not None:
        esf_by_state_grb[state] = v.X

print("Gurobi per-state ESF:", esf_by_state_grb)

### Infeasibility Checks

In [ ]:
# m = esf_gurobi_model
# for j in d_vars.values():
#     j.lb = 0.0
#     j.ub = 0.0
# 
# m.Params.DualReductions = 0
# m.optimize()
# print(m.Status)
# 
# m.computeIIS()
# m.write("fixed_zero.ilp")

In [ ]:
# m_fix.Params.DualReductions = 0
# m_fix.optimize()
# print(m_fix.Status)

In [ ]:
# m_fix.computeIIS()

In [ ]:
# m_fix.write('fixed_design.ilp')

In [ ]:
# state = (1,1,0)
# data = prepared_data_by_state[state]
# 
# print("sol_list length:", len(data["sol_list"]))
# print("filtered_solutions length:", len(data["filtered_solutions"]))
# 
# print("\nRaw theta bounds by original stage:")
# for k, tb in enumerate(data["theta_bounds_list"]):
#     print(f"stage {k}:")
#     print(tb)
# 
# print("\nFiltered theta bounds by filtered stage:")
# for k, tb in enumerate(data["filtered_theta_bounds"]):
#     print(f"filtered stage {k}:")
#     print(tb)

In [ ]:
# esf_gurobi_model.reset()
# esf_gurobi_model.optimize()
# print(esf_gurobi_model.Status)   # expect GRB.INFEASIBLE here

In [ ]:
# esf_gurobi_model.computeIIS()
# esf_gurobi_model.write("smolyak_infeasible.ilp")

In [ ]:
# esf_gurobi_model.setObjective(0.0, GRB.MINIMIZE)      # pure feasibility
# esf_gurobi_model.Params.DualReductions = 0
# esf_gurobi_model.reset()
# esf_gurobi_model.optimize()

In [ ]:
# print("Status with fixed design:", esf_gurobi_model.Status)

In [ ]:
# m_relax = esf_gurobi_model.copy()
# m_relax.feasRelaxS(0, False, True, True)
# m_relax.optimize()

### Results

In [ ]:
d_vars

In [ ]:
for i, dvar in d_vars.items():
    print(f"Design variable {i}: {dvar.X}")

In [ ]:
ESF.X

In [ ]:
esf_gurobi_model.ObjVal

## Validation

In [35]:
# design_vector = np.array([d_vars[j].X for j in range(nd)])
# design_vector = np.array(fixed_d)
# design_vector = np.array(d_v_lb)
design_vector = np.array([10, 10, 0.68])

In [36]:
design_vector

array([10.  , 10.  ,  0.68])

In [37]:
esf_gl, gl_esf_by_state = calculate_gl_esf(y_d=y_dict, prepared_data_by_state=load_prepared_data_by_state, d_v=design_vector, n_q=16)

All filtered solutions are None for state (0, 0, 0); skipping state
Skipping Gaussian SF for state (0, 0, 1): No region found that contains the given t_vector.
Finished for state (0, 0, 1).
All filtered solutions are None for state (0, 1, 0); skipping state
All filtered solutions are None for state (1, 0, 0); skipping state
Skipping Gaussian SF for state (0, 1, 1): No region found that contains the given t_vector.
Finished for state (0, 1, 1).
Skipping Gaussian SF for state (1, 0, 1): No region found that contains the given t_vector.
Finished for state (1, 0, 1).
Skipping Gaussian SF for state (1, 1, 0): No region found that contains the given t_vector.
Finished for state (1, 1, 0).
Gaussian Legendre Stochastic Flexibility Elapsed time: 0.0050 s
Finished for state (1, 1, 1).


In [38]:
print(f'Gauss Legendre ESF: {esf_gl}')

Gauss Legendre ESF: 0.4260452492070285


In [39]:
gl_esf_by_state

{(1, 1, 1): np.float64(0.5605858542197744)}

In [40]:
esf_sm, sm_esf_by_state = calculate_sm_esf(y_dict, load_prepared_data_by_state, design_vector, s_level=16)

All filtered solutions are None for state (0, 0, 0); skipping state
Skipping Smolyak SF for state (0, 0, 1): The provided theta_vector is not inside any critical region of the solution.
Finished for state (0, 0, 1).
All filtered solutions are None for state (0, 1, 0); skipping state
All filtered solutions are None for state (1, 0, 0); skipping state
Skipping Smolyak SF for state (0, 1, 1): The provided theta_vector is not inside any critical region of the solution.
Finished for state (0, 1, 1).
Skipping Smolyak SF for state (1, 0, 1): The provided theta_vector is not inside any critical region of the solution.
Finished for state (1, 0, 1).
Skipping Smolyak SF for state (1, 1, 0): The provided theta_vector is not inside any critical region of the solution.
Finished for state (1, 1, 0).
Smolyak stochastic flexibility computed in 0.1405 seconds.
Finished for state (1, 1, 1).


In [41]:
print(f'Smolyak ESF: {esf_sm}')

Smolyak ESF: 0.42696251431772336


In [ ]:
esf_sm_approx, sm_esf_by_state_approx = calculate_sm_esf(y_dict, load_prepared_data_by_state, design_vector, s_level=4, ns_u=nodes_u, ws_u=weights_u)

In [ ]:
esf_sm_approx

In [ ]:
print("Python per-state ESF:", sm_esf_by_state_approx)

In [ ]:
esf_by_state_grb

In [ ]:
esf_ub = ESF.X
esf_lb = esf_sm

In [ ]:
rel_error = abs(esf_ub-esf_lb)/abs(esf_ub + 1e-6) *100

In [ ]:
rel_error

## Diagnostics

In [ ]:
# test_state = (1, 1, 1)

In [ ]:
# block_map[test_state]['node_on']

In [ ]:
# for n in range(nodes_u.shape[0]):
#     print(f"{n}: {m_fix.getVarByName(f'sf_{test_state}_node_on[{n}]').X}")

In [ ]:
# block_map.keys()

In [ ]:
# node_on_val = float(nodeon[n].X)
# node_on_val = m_fix.getVarByName(f"sf_{test_state}_node_on[{23}]").X

In [ ]:
# node_on_val

In [ ]:
# for sta in y_dict:
#     try:
#         py_nodes = debug_python_smolyak_nodes_for_state(sta, design_vector, nodes_u, weights_u)
#         for idx, it in enumerate(py_nodes):
#             if not it['feasible']:
#                 print(f"{sta} \t {idx} \t {it['feasible']}")
#     except ValueError:
#         print(f'Skipping state {sta}')

In [ ]:
# load_prepared_data_by_state[(1,1,0)]['sol_list'][0].get_region(design_vector.reshape(-1,1))

In [ ]:
# grb_nodes = debug_gurobi_smolyak_nodes_for_state(m_fix, block_map, test_state, nodes_u, weights_u, state_label="S_111")

In [ ]:
# summary = compare_py_vs_gurobi_nodes(py_nodes, grb_nodes)

## Exporting

In [ ]:
results_dict = (
    {f'd{j}': d_vars[j].X for j in range(nd)}
    | {
        'ESF': ESF.X,
        'Obj': esf_gurobi_model.ObjVal,
        'ESF_actual': esf_sm,
        'rel_error': rel_error,
        'solver_runtime': esf_gurobi_model.Runtime,
        "NumVars": esf_gurobi_model.NumVars,
        "NumConstrs": esf_gurobi_model.NumConstrs,
        "NumGenConstrs": esf_gurobi_model.NumGenConstrs,
        "NumTotVars": esf_gurobi_model.NumVars,
        "NumBinVars": esf_gurobi_model.NumBinVars,
        "NumIntVars": esf_gurobi_model.NumIntVars,
        "NumContVars": esf_gurobi_model.NumVars - esf_gurobi_model.NumIntVars,
    }
)

In [ ]:
with open(f"pkl_output\MIPGap_{int(MIP_gap*100):03d}\\results_{int(esf_target*100):03d}_{quad_method}_{smolyak_level}.pkl", 'wb') as file:
    pickle.dump(results_dict, file)

In [ ]:
with open(f"pkl_output\MIPGap_{int(MIP_gap*100):03d}\\results_{int(esf_target*100):03d}_{quad_method}_{smolyak_level}.pkl", 'rb') as file:
    load_results = pickle.load(file)

In [ ]:
final_iter_dict = {
    "runtime_sec": esf_gurobi_model.Runtime,
    "nodecnt": esf_gurobi_model.NodeCount,
    "itcnt": esf_gurobi_model.IterCount,
    "obj_best": esf_gurobi_model.ObjVal if esf_gurobi_model.SolCount > 0 else None,
    "obj_bound": esf_gurobi_model.ObjBound,
    "gap_rel": esf_gurobi_model.MIPGap if esf_gurobi_model.SolCount > 0 else None,
    "gap_pct": 100.0 * esf_gurobi_model.MIPGap if esf_gurobi_model.SolCount > 0 else None,
    "solcnt": esf_gurobi_model.SolCount,
    "source": "final_model_state",
}

In [ ]:
hist_target.append(final_iter_dict)

In [ ]:
with open(f'pkl_output\MIPGap_{int(MIP_gap*100):03d}\solve_history_{int(esf_target*100):03d}_{quad_method}_{smolyak_level}.pkl', 'wb') as file:
    pickle.dump(hist_target, file)

In [ ]:
with open(f'pkl_output\MIPGap_{int(MIP_gap*100):03d}\solve_history_{int(esf_target*100):03d}_{quad_method}_{smolyak_level}.pkl', 'rb') as file:
    load_history = pickle.load(file)

## Plotting

In [ ]:
df_target = pd.DataFrame(load_history)

In [ ]:
df_target.loc[df_target.index[:-1], "source"] = "callback"

In [ ]:
df_target

In [ ]:
# Gap vs time
plot_df_gap = df_target.dropna(subset=["gap_pct"])

plt.figure(figsize=(7,5))
plt.plot(plot_df_gap["runtime_sec"], plot_df_gap["gap_pct"], marker="o", ms=3)
plt.xlabel("Time (s)")
plt.ylabel("Optimality gap (%)")
plt.xscale("log")
# plt.yscale("log")   # optional but often useful
plt.grid(True, alpha=0.3)
plt.title("Gurobi optimality gap vs time")
plt.show()

In [ ]:
# Gap vs iterations
plot_df_gap = df_target.dropna(subset=["gap_pct"])

plt.figure(figsize=(7,5))
plt.plot(plot_df_gap["itcnt"], plot_df_gap["gap_pct"], marker="o", ms=3)
plt.xlabel("Simplex iterations")
plt.ylabel("Optimality gap (%)")
plt.xscale("log")
# plt.yscale("log")
plt.grid(True, alpha=0.3)
plt.title("Gurobi optimality gap vs iterations")
plt.show()

In [ ]:
plot_df_bds = df_target.dropna(subset=["runtime_sec", "obj_best", "obj_bound"])

plt.figure(figsize=(7,5))
plt.plot(plot_df_bds["runtime_sec"], plot_df_bds["obj_best"], label="Upper bound")
plt.plot(plot_df_bds["runtime_sec"], plot_df_bds["obj_bound"], label="Lower bound")
plt.xlabel("Time (s)")
plt.ylabel("Objective bound")
plt.title("Bounds vs time")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

In [ ]:
plot_df_bds = df_target.dropna(subset=["itcnt", "obj_best", "obj_bound"])

plt.figure(figsize=(7,5))
plt.plot(plot_df_bds["itcnt"], plot_df_bds["obj_best"], label="Upper bound")
plt.plot(plot_df_bds["itcnt"], plot_df_bds["obj_bound"], label="Lower bound")
plt.xlabel("Iterations")
plt.ylabel("Objective bound")
plt.title("Bounds vs iterations")
plt.xscale("log")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()